In [9]:
import math
import matplotlib.pyplot as plt
from matplotlib.widgets import TextBox, Button
import numpy as np

TEAL = '#0D9488'
AMBER = '#F59E0B'
PURPLE = '#8B5CF6'
BLUE = '#3B82F6'
RED = '#EF4444'
GRAY = '#94A3B8'

plt.rcParams.update({
    'figure.figsize': (11, 6),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.labelsize': 11,
    'axes.titlesize': 13,
    'font.family': 'DejaVu Sans',
})

N = m1g cos θ
Fmotriz = m2g − m1g sin θ

Se |Fmotriz| ≤ µsN : a = 0 (EQUILÍBRIO ESTÁTICO)

Senão: a =
Fmotriz − µkN sinal(Fmotriz)
m1 + m2
(MOVIMENTO)

In [10]:
def calcular_aceleracao_sistema(massa_bloco_plano, massa_bloco_suspenso, angulo_graus, coeficiente_estatico, coeficiente_cinetico, aceleracao_gravidade=9.81):

    angulo_radianos = np.radians(angulo_graus) #é para converter graus para radianos

    # Força normal=(N = m₁ • g • cos(θ)
    forca_normal = massa_bloco_plano * aceleracao_gravidade * np.cos(angulo_radianos)

    # Força motriz
    forca_motriz = (massa_bloco_suspenso * aceleracao_gravidade) - (massa_bloco_plano * aceleracao_gravidade * np.sin(angulo_radianos))

    # max do limite atrito estatico
    limite_atrito_estatico = coeficiente_estatico * forca_normal

    #se força do amortriz for menor ou igual o sistema fica parado
    if np.abs(forca_motriz) <= limite_atrito_estatico:
        aceleracao = 0.0
        estado_movimento = "Equilíbrio Estático (aceleração = 0)"
    # se não entra em movimento
    else:
        sinal_forca = np.sign(forca_motriz)
        forca_atrito_cinetico = coeficiente_cinetico * forca_normal
        aceleracao = (forca_motriz - (forca_atrito_cinetico * sinal_forca)) / (massa_bloco_plano + massa_bloco_suspenso)
        estado_movimento = "Em Movimento"

    return aceleracao, forca_normal, forca_motriz, estado_movimento

In [11]:
def simular_plano_inclinado(massa_bloco_plano, massa_bloco_suspenso, angulo_graus, coeficiente_estatico, coeficiente_cinetico, tempo_maximo=5.0, numero_pontos=200):

    aceleracao, forca_normal, forca_motriz, estado_movimento = calcular_aceleracao_sistema(
        massa_bloco_plano, massa_bloco_suspenso, angulo_graus, coeficiente_estatico, coeficiente_cinetico
    )
    #vetr tempo cria lista de instantes de tempo
    vetor_tempo = np.linspace(0, tempo_maximo, numero_pontos)
    posicao_inicial = 0.0
    velocidade_inicial = 0.0

    # Fórmulas de MRUV: x(t) = x0 + v0*t + 0.5*a*t^2 e v(t) = v0 + a*t
    #calcula posição e velocidade
    posicao_ao_longo_tempo = posicao_inicial + velocidade_inicial * vetor_tempo + 0.5 * aceleracao * (vetor_tempo ** 2)
    velocidade_ao_longo_tempo = velocidade_inicial + aceleracao * vetor_tempo

    # calcula tensão do fio
    gravidade = 9.81
    tracao_no_fio = massa_bloco_suspenso * (gravidade - aceleracao)

    return {
        "tempo": vetor_tempo,
        "posicao": posicao_ao_longo_tempo,
        "velocidade": velocidade_ao_longo_tempo,
        "tracao": tracao_no_fio,
        "aceleracao": aceleracao,
        "estado": estado_movimento,
        "normal": forca_normal,
        "forca_motriz": forca_motriz
    }

 $\omega = \frac{v}{R}$, $x(t) = R \cos(\omega t)$, $y(t) = R \sin(\omega t)$ e $F_c = m \omega^2 R$):

In [12]:
def simular_movimento_circular_uniforme(massa_corpo, raio_circunferencia, velocidade_tangencial, tempo_maximo=5.0, numero_pontos=200):

    vetor_tempo = np.linspace(0, tempo_maximo, numero_pontos)

    # Velocidade angular (omega = v / R). Protege contra divisão por zero se o raio for 0.
    velocidade_angular = velocidade_tangencial / raio_circunferencia if raio_circunferencia > 0 else 0.0

    # usa posição x e y,cossenos e senos multiplicado pelo raio para mapear coordenadas
    fase_angular = velocidade_angular * vetor_tempo
    posicao_x = raio_circunferencia * np.cos(fase_angular)
    posicao_y = raio_circunferencia * np.sin(fase_angular)

    # Força Centrípeta: Fc = m * ω² * R
    #Calcula a força necessária para manter o corpo a girar em trajetória circula
    forca_centripeta = massa_corpo * (velocidade_angular ** 2) * raio_circunferencia

    return {
        "tempo": vetor_tempo,
        "posicao_x": posicao_x,
        "posicao_y": posicao_y,
        "velocidade_angular": velocidade_angular,
        "forca_centripeta": forca_centripeta
    }

In [17]:

if __name__ == "__main__":
    print("=== Simulação do Plano Inclinado ===")
    resultado_plano = simular_plano_inclinado(
        massa_bloco_plano=6.0,
        massa_bloco_suspenso=6.0,
        angulo_graus=30.0,
        coeficiente_estatico=0.2,
        coeficiente_cinetico=0.1
    )
    print(f"Estado: {resultado_plano['estado']}")
    print(f"Aceleração calculada: {resultado_plano['aceleracao']:.2f} m/s²")
    print(f"Tração no fio: {resultado_plano['tracao']:.2f} N")

    print("\n=== Simulação do Movimento Circular Uniforme ===")
    resultado_mcu = simular_movimento_circular_uniforme(
        massa_corpo=2.0,
        raio_circunferencia=1.5,
        velocidade_tangencial=3.0
    )
    print(f"Velocidade Angular (omega): {resultado_mcu['velocidade_angular']:.2f} rad/s")
    print(f"Força Centrípeta: {resultado_mcu['forca_centripeta']:.2f} N")

=== Simulação do Plano Inclinado ===
Estado: Em Movimento
Aceleração calculada: 2.03 m/s²
Tração no fio: 46.69 N

=== Simulação do Movimento Circular Uniforme ===
Velocidade Angular (omega): 2.00 rad/s
Força Centrípeta: 12.00 N
